In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip -q install -U transformers datasets peft accelerate scikit-learn pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have 

In [3]:
%%writefile train_fedavg_tinybert_lora_imdb.py
import os
import gc
import json
import time
import glob
import shutil
import random
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel


@dataclass
class Config:
    # ===== Model / Data =====
    model_name: str = "huawei-noah/TinyBERT_General_4L_312D"
    dataset_name: str = "imdb"
    setting: str = "FedAvg + TinyBERT + LoRA"
    num_labels: int = 2
    seed: int = 42

    # ===== Output =====
    drive_root: str = "/content/drive/MyDrive/fedavg_tinybert_lora_imdb"
    experiment_name: str = "fedavg_tinybert_lora_imdb"

    # ===== Tokenization / Data =====
    max_length: int = 256
    validation_ratio: float = 0.1

    # ===== Federated Learning =====
    num_clients: int = 5
    local_epochs: int = 1
    local_batch_size: int = 16
    eval_batch_size: int = 64
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0

    # ===== Partition =====
    partition_type: str = "noniid"   # "iid" hoặc "noniid"
    dirichlet_alpha: float = 0.5

    # ===== LoRA =====
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    target_modules: tuple = ("query", "key", "value")

    # ===== Early stopping =====
    monitor: str = "accuracy"
    greater_is_better: bool = True
    early_stopping_patience: int = 3
    min_delta: float = 0.0

    # ===== Stability for Colab =====
    fp16: bool = False
    num_workers: int = 0
    resume: bool = True
    save_latest_k: int = 2

    # ===== Client evaluation =====
    evaluate_each_client: bool = True

    # ===== Safety upper bound =====
    max_rounds: int = 1000

    @property
    def exp_dir(self):
        return os.path.join(self.drive_root, self.experiment_name)

    @property
    def ckpt_dir(self):
        return os.path.join(self.exp_dir, "checkpoints")

    @property
    def best_model_dir(self):
        return os.path.join(self.exp_dir, "best_model")

    @property
    def final_model_dir(self):
        return os.path.join(self.exp_dir, "final_model")

    @property
    def history_json_path(self):
        return os.path.join(self.exp_dir, "history.json")

    @property
    def history_csv_path(self):
        return os.path.join(self.exp_dir, "history.csv")

    @property
    def client_history_json_path(self):
        return os.path.join(self.exp_dir, "client_history.json")

    @property
    def client_history_csv_path(self):
        return os.path.join(self.exp_dir, "client_history.csv")

    @property
    def config_path(self):
        return os.path.join(self.exp_dir, "config.json")

    @property
    def summary_path(self):
        return os.path.join(self.exp_dir, "run_summary.json")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dirs(cfg: Config):
    os.makedirs(cfg.drive_root, exist_ok=True)
    os.makedirs(cfg.exp_dir, exist_ok=True)
    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    os.makedirs(cfg.best_model_dir, exist_ok=True)
    os.makedirs(cfg.final_model_dir, exist_ok=True)


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_history_csv(path, history):
    if len(history) == 0:
        return
    pd.DataFrame(history).to_csv(path, index=False, encoding="utf-8")


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params


def metric_improved(current, best, greater_is_better=True, min_delta=0.0):
    if greater_is_better:
        return current > (best + min_delta)
    return current < (best - min_delta)


def tokenize_function(examples, tokenizer, max_length):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
    )


def dirichlet_noniid_partition(labels, num_clients=5, alpha=0.5, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.array(labels)
    unique_classes = np.unique(labels)

    client_indices = [[] for _ in range(num_clients)]

    for c in unique_classes:
        idx = np.where(labels == c)[0]
        rng.shuffle(idx)

        proportions = rng.dirichlet(np.repeat(alpha, num_clients))
        split_points = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
        splits = np.split(idx, split_points)

        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())

    for client_id in range(num_clients):
        rng.shuffle(client_indices[client_id])

    return client_indices


def iid_partition(num_samples, num_clients=5, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.arange(num_samples)
    rng.shuffle(indices)
    splits = np.array_split(indices, num_clients)
    return [split.tolist() for split in splits]


def prepare_data(cfg: Config, tokenizer):
    ds = load_dataset(cfg.dataset_name)

    split_ds = ds["train"].train_test_split(
        test_size=cfg.validation_ratio,
        seed=cfg.seed,
        shuffle=True
    )

    train_ds = split_ds["train"]
    val_ds = split_ds["test"]

    tokenized_train = train_ds.map(
        lambda x: tokenize_function(x, tokenizer, cfg.max_length),
        batched=True,
        remove_columns=["text"],
    )
    tokenized_val = val_ds.map(
        lambda x: tokenize_function(x, tokenizer, cfg.max_length),
        batched=True,
        remove_columns=["text"],
    )

    tokenized_train = tokenized_train.rename_column("label", "labels")
    tokenized_val = tokenized_val.rename_column("label", "labels")

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    labels = tokenized_train["labels"]

    if cfg.partition_type.lower() == "noniid":
        client_splits = dirichlet_noniid_partition(
            labels=labels,
            num_clients=cfg.num_clients,
            alpha=cfg.dirichlet_alpha,
            seed=cfg.seed
        )
    else:
        client_splits = iid_partition(
            num_samples=len(tokenized_train),
            num_clients=cfg.num_clients,
            seed=cfg.seed
        )

    client_loaders = []
    client_num_samples = []

    for client_id in range(cfg.num_clients):
        subset = Subset(tokenized_train, client_splits[client_id])
        loader = DataLoader(
            subset,
            batch_size=cfg.local_batch_size,
            shuffle=True,
            collate_fn=collator,
            num_workers=cfg.num_workers,
            pin_memory=torch.cuda.is_available(),
        )
        client_loaders.append(loader)
        client_num_samples.append(len(client_splits[client_id]))

    val_loader = DataLoader(
        tokenized_val,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return client_loaders, client_num_samples, val_loader


def create_base_peft_model(cfg: Config):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=cfg.num_labels,
        ignore_mismatched_sizes=True
    )

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        target_modules=list(cfg.target_modules),
        modules_to_save=["classifier"],
    )

    model = get_peft_model(base_model, peft_config)
    return model


def load_peft_model_from_dir(cfg: Config, adapter_dir: str, device: str):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=cfg.num_labels,
        ignore_mismatched_sizes=True
    )
    model = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=True)
    model.to(device)
    return model


def extract_lora_state_dict(model):
    state = model.state_dict()
    lora_state = {}
    for k, v in state.items():
        if ("lora_" in k) or ("modules_to_save" in k):
            lora_state[k] = v.detach().cpu().clone()
    return lora_state


def load_lora_state_dict(model, lora_state_dict):
    model.load_state_dict(lora_state_dict, strict=False)


def average_lora_states(client_states, client_weights):
    if len(client_states) == 0:
        raise ValueError("client_states is empty.")

    avg_state = {}
    keys = client_states[0].keys()

    total_weight = float(sum(client_weights))
    if total_weight <= 0:
        raise ValueError("Total client weights must be > 0.")

    norm_weights = [w / total_weight for w in client_weights]

    for k in keys:
        avg_tensor = None
        for state, w in zip(client_states, norm_weights):
            tensor = state[k].float() * w
            if avg_tensor is None:
                avg_tensor = tensor
            else:
                avg_tensor += tensor
        avg_state[k] = avg_tensor
    return avg_state


def compute_comm_cost_mb(lora_state_dict, num_clients):
    total_bytes_one_way = 0
    for _, tensor in lora_state_dict.items():
        total_bytes_one_way += tensor.numel() * tensor.element_size()
    total_bytes_round = total_bytes_one_way * num_clients * 2
    return total_bytes_round / (1024 ** 2)


def list_checkpoints(ckpt_dir: str):
    ckpts = []
    if not os.path.exists(ckpt_dir):
        return ckpts

    for path in glob.glob(os.path.join(ckpt_dir, "checkpoint-round-*")):
        try:
            round_num = int(path.split("-")[-1])
            ckpts.append((round_num, path))
        except Exception:
            pass

    ckpts.sort(key=lambda x: x[0])
    return ckpts


def latest_checkpoint_path(ckpt_dir: str):
    ckpts = list_checkpoints(ckpt_dir)
    return ckpts[-1][1] if ckpts else None


def prune_old_checkpoints(ckpt_dir: str, keep_latest_k: int = 2):
    ckpts = list_checkpoints(ckpt_dir)
    while len(ckpts) > keep_latest_k:
        _, old_path = ckpts.pop(0)
        try:
            shutil.rmtree(old_path)
        except Exception:
            pass


def clear_dir_keep_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def save_training_artifacts(cfg: Config, save_dir: str, global_lora_state, tokenizer):
    clear_dir_keep_dir(save_dir)

    torch.save(global_lora_state, os.path.join(save_dir, "global_lora_state.pt"))

    save_model = create_base_peft_model(cfg)
    load_lora_state_dict(save_model, global_lora_state)
    save_model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    merged_dir = os.path.join(save_dir, "merged_model")
    os.makedirs(merged_dir, exist_ok=True)
    merged = save_model.merge_and_unload()
    merged.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)

    save_json(os.path.join(save_dir, "model_info.json"), {
        "model_name": cfg.model_name,
        "dataset_name": cfg.dataset_name,
        "setting": cfg.setting,
        "num_labels": cfg.num_labels,
    })

    del save_model
    del merged
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(
    cfg,
    round_idx,
    global_lora_state,
    tokenizer,
    best_metric,
    patience_counter,
    history,
    client_history
):
    ckpt_path = os.path.join(cfg.ckpt_dir, f"checkpoint-round-{round_idx}")
    os.makedirs(ckpt_path, exist_ok=True)

    torch.save(global_lora_state, os.path.join(ckpt_path, "global_lora_state.pt"))

    ckpt_model = create_base_peft_model(cfg)
    load_lora_state_dict(ckpt_model, global_lora_state)
    ckpt_model.save_pretrained(ckpt_path)
    tokenizer.save_pretrained(ckpt_path)

    state = {
        "round": round_idx,
        "best_metric": best_metric,
        "patience_counter": patience_counter,
        "history": history,
        "client_history": client_history,
        "config": asdict(cfg),
    }
    torch.save(state, os.path.join(ckpt_path, "training_state.pt"))

    with open(os.path.join(cfg.ckpt_dir, "latest_checkpoint.txt"), "w", encoding="utf-8") as f:
        f.write(ckpt_path)

    prune_old_checkpoints(cfg.ckpt_dir, cfg.save_latest_k)

    del ckpt_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return ckpt_path


def load_latest_checkpoint(cfg, device="cpu"):
    ckpt_path = latest_checkpoint_path(cfg.ckpt_dir)
    if ckpt_path is None or not cfg.resume:
        return None, 0, None, 0, [], []

    lora_state_path = os.path.join(ckpt_path, "global_lora_state.pt")
    state_path = os.path.join(ckpt_path, "training_state.pt")
    adapter_config_path = os.path.join(ckpt_path, "adapter_config.json")

    if not (os.path.exists(lora_state_path) and os.path.exists(state_path) and os.path.exists(adapter_config_path)):
        print("Checkpoint files missing. Start from scratch.")
        return None, 0, None, 0, [], []

    print(f"Resuming from checkpoint: {ckpt_path}")

    server_model = load_peft_model_from_dir(cfg, ckpt_path, device=device)
    state = torch.load(state_path, map_location=device)

    return (
        server_model,
        state["round"],
        state.get("best_metric", None),
        state.get("patience_counter", 0),
        state.get("history", []),
        state.get("client_history", []),
    )


@torch.no_grad()
def evaluate(model, loader, device, use_amp=False, split_name="validation"):
    model.eval()

    total_loss = 0.0
    total_steps = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc=f"Evaluating {split_name}", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}

        if use_amp:
            with torch.cuda.amp.autocast(enabled=True):
                outputs = model(**batch)
                loss = outputs.loss
                logits = outputs.logits
        else:
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

        preds = torch.argmax(logits, dim=-1)

        total_loss += loss.item()
        total_steps += 1
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(batch["labels"].detach().cpu().numpy().tolist())

    return {
        "eval_loss": float(total_loss / max(total_steps, 1)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
        "macro_f1": float(f1_score(all_labels, all_preds, average="macro")),
    }


def train_one_client(cfg, global_lora_state, client_loader, client_id, num_samples, device, eval_loader):
    client_model = create_base_peft_model(cfg).to(device)
    load_lora_state_dict(client_model, global_lora_state)

    total_params, trainable_params = count_parameters(client_model)

    use_amp = cfg.fp16 and device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    optimizer = torch.optim.AdamW(
        [p for p in client_model.parameters() if p.requires_grad],
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay
    )

    client_model.train()
    client_start = time.time()
    running_loss = 0.0
    total_steps = 0

    for local_epoch in range(cfg.local_epochs):
        pbar = tqdm(
            client_loader,
            desc=f"Client {client_id} epoch {local_epoch + 1}/{cfg.local_epochs}",
            leave=False
        )

        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.cuda.amp.autocast(enabled=True):
                    outputs = client_model(**batch)
                    loss = outputs.loss

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = client_model(**batch)
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), cfg.max_grad_norm)
                optimizer.step()

            running_loss += loss.item()
            total_steps += 1

            if total_steps % 20 == 0:
                pbar.set_postfix(loss=f"{loss.item():.4f}")

    client_train_time = time.time() - client_start
    avg_train_loss = float(running_loss / max(total_steps, 1))

    if cfg.evaluate_each_client:
        client_eval = evaluate(
            client_model,
            eval_loader,
            device=device,
            use_amp=use_amp,
            split_name=f"client_{client_id}_validation"
        )
    else:
        client_eval = {
            "eval_loss": None,
            "accuracy": None,
            "macro_f1": None,
        }

    client_lora_state = extract_lora_state_dict(client_model)

    del client_model
    del optimizer
    del scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "client_state": client_lora_state,
        "num_samples": int(num_samples),
        "train_time": float(client_train_time),
        "train_loss": float(avg_train_loss),
        "eval_loss": None if client_eval["eval_loss"] is None else float(client_eval["eval_loss"]),
        "accuracy": None if client_eval["accuracy"] is None else float(client_eval["accuracy"]),
        "macro_f1": None if client_eval["macro_f1"] is None else float(client_eval["macro_f1"]),
        "trainable_params": int(trainable_params),
        "total_params": int(total_params),
    }


def verify_required_outputs(cfg: Config):
    required_paths = [
        cfg.history_json_path,
        cfg.history_csv_path,
        cfg.client_history_json_path,
        cfg.client_history_csv_path,
        cfg.summary_path,
        cfg.config_path,
        os.path.join(cfg.best_model_dir, "adapter_config.json"),
        os.path.join(cfg.best_model_dir, "global_lora_state.pt"),
        os.path.join(cfg.best_model_dir, "merged_model", "config.json"),
        os.path.join(cfg.final_model_dir, "adapter_config.json"),
        os.path.join(cfg.final_model_dir, "global_lora_state.pt"),
        os.path.join(cfg.final_model_dir, "merged_model", "config.json"),
    ]

    missing = [p for p in required_paths if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError("Missing required output files:\n" + "\n".join(missing))


def train():
    cfg = Config()
    set_seed(cfg.seed)
    ensure_dirs(cfg)
    save_json(cfg.config_path, asdict(cfg))

    device = get_device()
    print("=" * 100)
    print("Device:", device)
    print("Experiment dir:", cfg.exp_dir)
    print("=" * 100)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)

    client_loaders, client_num_samples, val_loader = prepare_data(cfg, tokenizer)

    resumed_model, start_round, best_metric, patience_counter, history, client_history = load_latest_checkpoint(
        cfg, device=device
    )

    if resumed_model is not None:
        server_model = resumed_model
    else:
        server_model = create_base_peft_model(cfg).to(device)
        start_round = 0
        best_metric = None
        patience_counter = 0
        history = []
        client_history = []

    total_params, trainable_params = count_parameters(server_model)

    if best_metric is None:
        best_metric = -1e18 if cfg.greater_is_better else 1e18

    print(f"Start from round {start_round + 1}")
    print(f"Best {cfg.monitor} so far: {best_metric:.6f}")
    print(f"Patience counter: {patience_counter}")
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")

    stopped_early = False
    round_idx = start_round + 1

    while round_idx <= cfg.max_rounds:
        round_start = time.time()

        # ===== server broadcast =====
        global_lora_state = extract_lora_state_dict(server_model)

        client_states = []
        client_weights = []
        client_times = []
        round_client_logs = []

        # ===== client local training =====
        for client_id in range(cfg.num_clients):
            print(f"\n[Round {round_idx}] Training client {client_id + 1}/{cfg.num_clients}")

            client_result = train_one_client(
                cfg=cfg,
                global_lora_state=global_lora_state,
                client_loader=client_loaders[client_id],
                client_id=client_id,
                num_samples=client_num_samples[client_id],
                device=device,
                eval_loader=val_loader
            )

            client_states.append(client_result["client_state"])
            client_weights.append(client_result["num_samples"])
            client_times.append(client_result["train_time"])

            client_log = {
                "round": int(round_idx),
                "accuracy": client_result["accuracy"],
                "macro_f1": client_result["macro_f1"],
                "eval_loss": client_result["eval_loss"],
                "round_time": None,
                "client_train_time": float(client_result["train_time"]),
                "communication_cost_MB": None,
                "trainable_params": int(client_result["trainable_params"]),
                "total_params": int(client_result["total_params"]),
                "model_name": cfg.model_name,
                "dataset_name": cfg.dataset_name,
                "setting": cfg.setting,
                "num_clients": int(cfg.num_clients),
                "local_epochs": int(cfg.local_epochs),
                "partition_type": cfg.partition_type,
                "best_metric_so_far": None,
                "patience_counter": None,
                "is_new_best": None,
                "client_id": int(client_id),
                "num_samples": int(client_result["num_samples"]),
                "train_loss": float(client_result["train_loss"]),
            }

            round_client_logs.append(client_log)

        # ===== server aggregation =====
        aggregated_lora_state = average_lora_states(client_states, client_weights)
        load_lora_state_dict(server_model, aggregated_lora_state)

        # ===== server validation =====
        use_amp = cfg.fp16 and device == "cuda"
        val_metrics = evaluate(
            server_model,
            val_loader,
            device=device,
            use_amp=use_amp,
            split_name="validation"
        )

        round_time = time.time() - round_start
        client_train_time_total = float(sum(client_times))
        communication_cost_MB = float(compute_comm_cost_mb(global_lora_state, cfg.num_clients))

        current_metric = val_metrics[cfg.monitor]
        is_new_best = metric_improved(
            current=current_metric,
            best=best_metric,
            greater_is_better=cfg.greater_is_better,
            min_delta=cfg.min_delta
        )

        if is_new_best:
            best_metric = current_metric
            patience_counter = 0
            save_training_artifacts(
                cfg=cfg,
                save_dir=cfg.best_model_dir,
                global_lora_state=aggregated_lora_state,
                tokenizer=tokenizer
            )
        else:
            patience_counter += 1

        round_result = {
            "round": int(round_idx),
            "accuracy": float(val_metrics["accuracy"]),
            "macro_f1": float(val_metrics["macro_f1"]),
            "eval_loss": float(val_metrics["eval_loss"]),
            "round_time": float(round_time),
            "client_train_time": float(client_train_time_total),
            "communication_cost_MB": float(communication_cost_MB),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": cfg.model_name,
            "dataset_name": cfg.dataset_name,
            "setting": cfg.setting,
            "num_clients": int(cfg.num_clients),
            "local_epochs": int(cfg.local_epochs),
            "partition_type": cfg.partition_type,
            "best_metric_so_far": float(best_metric),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }

        history.append(round_result)

        for client_log in round_client_logs:
            client_log["round_time"] = float(round_time)
            client_log["communication_cost_MB"] = float(communication_cost_MB)
            client_log["best_metric_so_far"] = float(best_metric)
            client_log["patience_counter"] = int(patience_counter)
            client_log["is_new_best"] = bool(is_new_best)
            client_history.append(client_log)

        save_json(cfg.history_json_path, history)
        save_history_csv(cfg.history_csv_path, history)

        save_json(cfg.client_history_json_path, client_history)
        save_history_csv(cfg.client_history_csv_path, client_history)

        ckpt_path = save_checkpoint(
            cfg=cfg,
            round_idx=round_idx,
            global_lora_state=aggregated_lora_state,
            tokenizer=tokenizer,
            best_metric=best_metric,
            patience_counter=patience_counter,
            history=history,
            client_history=client_history
        )

        print("-" * 100)
        print("SERVER ROUND LOG:")
        print(json.dumps(round_result, indent=2))
        print("CLIENT ROUND LOGS:")
        print(json.dumps(round_client_logs, indent=2))
        print("Checkpoint saved:", ckpt_path)
        print("-" * 100)

        if patience_counter >= cfg.early_stopping_patience:
            print(f"Early stopping triggered: no improvement in {cfg.early_stopping_patience} consecutive rounds.")
            stopped_early = True
            break

        round_idx += 1

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_lora_state = extract_lora_state_dict(server_model)
    save_training_artifacts(
        cfg=cfg,
        save_dir=cfg.final_model_dir,
        global_lora_state=final_lora_state,
        tokenizer=tokenizer
    )

    run_summary = {
        "model_name": cfg.model_name,
        "dataset_name": cfg.dataset_name,
        "setting": cfg.setting,
        "monitor_metric": cfg.monitor,
        "best_metric_so_far": float(best_metric),
        "num_rounds_completed": int(len(history)),
        "num_client_logs": int(len(client_history)),
        "stopped_early": bool(stopped_early),
        "best_model_dir": cfg.best_model_dir,
        "final_model_dir": cfg.final_model_dir,
        "latest_checkpoint": latest_checkpoint_path(cfg.ckpt_dir),
        "history_json": cfg.history_json_path,
        "history_csv": cfg.history_csv_path,
        "client_history_json": cfg.client_history_json_path,
        "client_history_csv": cfg.client_history_csv_path,
    }
    save_json(cfg.summary_path, run_summary)

    verify_required_outputs(cfg)

    print("=" * 100)
    print("OUTPUT DIRECTORY          :", cfg.exp_dir)
    print("Best model dir            :", cfg.best_model_dir)
    print("Final model dir           :", cfg.final_model_dir)
    print("Checkpoints dir           :", cfg.ckpt_dir)
    print("Server history JSON       :", cfg.history_json_path)
    print("Server history CSV        :", cfg.history_csv_path)
    print("Client history JSON       :", cfg.client_history_json_path)
    print("Client history CSV        :", cfg.client_history_csv_path)
    print("Run summary               :", cfg.summary_path)
    print("=" * 100)
    print("Done.")


if __name__ == "__main__":
    train()

Writing train_fedavg_tinybert_lora_imdb.py


In [4]:
!python train_fedavg_tinybert_lora_imdb.py

Device: cuda
Experiment dir: /content/drive/MyDrive/fedavg_tinybert_lora_imdb/fedavg_tinybert_lora_imdb
config.json: 100% 409/409 [00:00<00:00, 1.53MB/s]
vocab.txt: 232kB [00:00, 15.5MB/s]
README.md: 7.81kB [00:00, 15.5MB/s]
plain_text/train-00000-of-00001.parquet: 100% 21.0M/21.0M [00:01<00:00, 14.2MB/s]
plain_text/test-00000-of-00001.parquet: 100% 20.5M/20.5M [00:00<00:00, 31.4MB/s]
plain_text/unsupervised-00000-of-00001.p(…): 100% 42.0M/42.0M [00:00<00:00, 50.7MB/s]
Generating train split: 100% 25000/25000 [00:00<00:00, 71770.43 examples/s]
Generating test split: 100% 25000/25000 [00:00<00:00, 82657.85 examples/s]
Generating unsupervised split: 100% 50000/50000 [00:00<00:00, 127953.06 examples/s]
Map: 100% 22500/22500 [00:19<00:00, 1132.25 examples/s]
Map: 100% 2500/2500 [00:02<00:00, 1052.09 examples/s]
pytorch_model.bin: 100% 62.7M/62.7M [00:01<00:00, 44.3MB/s]
Loading weights: 100% 71/71 [00:00<00:00, 14640.16it/s]
BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyB